data loading


In [ ]:
import kagglehub
import pandas as pd
import os
import random
from datasets import Dataset
import datasets 
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# Download and get the dataset path
path = kagglehub.dataset_download("shanegerami/ai-vs-human-text")
csv_path = os.path.join(path, "AI_Human.csv")

# Load into DataFrame
df = pd.read_csv(csv_path)






data cleaning 


In [ ]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
class Text:
    def __init__(self, text, generated):
        self.generated = generated
        self.text = text

# Create a list of Text objects from the DataFrame
data = [Text(row['text'], row['generated']) for _, row in df.iterrows()]

#So, for _, row in df.iterrows() means "for each row in the DataFrame, ignore the index, and use the row data."

ai_text = list(filter(lambda x : x.generated == 1, data))
human_text = list(filter(lambda x : x.generated == 0 , data))
human_text_shrunk = human_text[:len(ai_text)]
data = ai_text + human_text_shrunk
random.shuffle(data)
output = [t.generated for t in data]
texts = [t.text for t in data]
new_df = pd.DataFrame({'text': texts, 'label': output})

# convert to a hugging face dataset
dataset = Dataset.from_pandas(new_df)




initalize the tockenizer and the model

In [ ]:
# Use a pipeline as a high-level helper
# from transformers import pipeline

# pipe = pipeline("fill-mask", model="distilbert/distilbert-base-uncased")(faster method but for testing model only not for fine tuning it )

# Load model directly


tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased",num_labels = 2)




Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


split data and tokenize data



In [ ]:

dataset = dataset.train_test_split(test_size=0.2)
train_ds = dataset['train']
test_ds = dataset['test']

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding=True)

tokenized_train = train_ds.map(tokenize, batched=True)
tokenized_test = test_ds.map(tokenize, batched=True)





# batch_size = 50000
# all_tokenized = []
# for i in range(0, len(data), batch_size):
#     texts = [item.text for item in data[i:i+batch_size]]
#     tokenized = tokenizer(
#         texts,
#         padding=True,
#         truncation=True,
#         return_tensors="pt"
#     )
#     all_tokenized.append(tokenized)




Map: 100%|██████████| 72576/72576 [00:37<00:00, 1912.05 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 72576
}) Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 290300
}) DistilBertTokenizerFast(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[